# Лабораторная 1. Языковая модель из счётчиков

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import math
import random
import re
import urllib.request

In [ ]:
url = 'https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/751b8e17d82a9765940a778f7a8c0d1086d8a9da/books_before/WarAndPeace.txt'
corpus_path = Path(".cache/WarAndPeace.txt")
corpus_path.parent.mkdir(exist_ok=True)

if not corpus_path.exists():
    urllib.request.urlretrieve(url, corpus_path)

text = corpus_path.read_text(encoding="utf-8-sig")
# Убираю строку с названием книги перед началом текста.
text = text.split("Л. Н. Толстой «Война и мир»", 1)[1].lower()

In [ ]:
# <unk> — специальный токен модели; в исходном корпусе его нет.
word_pattern = re.compile(r"<unk>|[а-яё]+(?:-[а-яё]+)*")

def tokenize(text):
    return word_pattern.findall(text.lower())

# Удаляю пунктуацию, числа и латиницу, чтобы моделировать только русские слова. дефис - на месте
words = tokenize(text)
frequencies = Counter(words)

print("Слов:", len(words))
print("Разных слов:", len(frequencies))
assert len(words) >= 100_000

print("\n20 самых частых слов:")
for word, count in frequencies.most_common(20):
    print(f"{word:15} {count:6}")

Слов: 433076
Разных слов: 48131

20 самых частых слов:
и                21084
в                10870
не                8520
что               7600
он                7395
на                6665
с                 5861
как               4011
его               3895
к                 3428
я                 2854
она               2732
но                2725
все               2578
было              2501
это               2434
то                2261
сказал            2000
так               1954
о                 1832


In [ ]:
split = int(len(words) * 0.9)
train = words[:split]
test = words[split:]

# Сохраняю исходный текст, чтобы потом взять из него предложение.
test_start = list(word_pattern.finditer(text))[split].start()
test_text = text[test_start:]
assert tokenize(test_text) == test

print("Обучение:", len(train))
print("Проверка:", len(test))

Обучение: 389768
Проверка: 43308


## Биграммная модель

Считаю слова и соседние пары только в обучающей части.
В знаменателе использую число исходящих переходов: последнее слово корпуса
не имеет следующего слова, поэтому оно не добавляет переход в свою строку.
При alpha=0 получаются обычные доли, при alpha>0 — сглаживание Лапласа.
Словарь фиксируется при обучении и дополняется одним токеном unk;
любое неизвестное слово при оценке отображается в этот токен.

In [ ]:
class BigramModel:
    def __init__(self, alpha=0.0):
        if not math.isfinite(alpha) or alpha < 0:
            raise ValueError("alpha должна быть конечной и неотрицательной")
        self.alpha = alpha
        self.unk = "<unk>"

    def fit(self, words):
        words = list(words)
        if not words:
            raise ValueError("Для обучения нужен непустой текст")
        self.word_counts = Counter(words)
        self.vocab = list(dict.fromkeys(words))
        if self.unk not in self.word_counts:
            self.vocab.append(self.unk)
        self.vocab_set = set(self.vocab)
        self.bigrams = defaultdict(Counter)
        for previous, word in zip(words, words[1:]):
            self.bigrams[previous][word] += 1

        # Последнее слово текста не имеет следующего слова.
        self.totals = {word: sum(row.values())
                       for word, row in self.bigrams.items()}
        return self

    def _known(self, word):
        return word if word in self.vocab_set else self.unk

    def prob(self, previous, word):
        previous, word = self._known(previous), self._known(word)
        count = self.bigrams.get(previous, {}).get(word, 0)
        total = self.totals.get(previous, 0)
        denominator = total + self.alpha * len(self.vocab)
        return (count + self.alpha) / denominator if denominator else 0.0

    def generate(self, start, length=30, temperature=1.0, seed=None):
        if length < 1 or not isinstance(length, int):
            raise ValueError("length должна быть положительным целым числом")
        if not math.isfinite(temperature) or temperature <= 0:
            raise ValueError("Температура должна быть конечной и положительной")
        start_words = tokenize(start)
        if len(start_words) != 1:
            raise ValueError("Нужно ровно одно стартовое слово")
        generated = [self._known(start_words[0])]
        rng = random.Random(seed)
        for _ in range(length - 1):
            row = self.bigrams.get(generated[-1], {})
            if self.alpha > 0:
                candidates = self.vocab
                weights = [row.get(word, 0) + self.alpha for word in candidates]
            elif row:
                candidates, weights = zip(*row.items())
            else:
                # Без сглаживания пустая строка не задаёт распределение.
                # Возобновляю генерацию по частотам слов обучения.
                candidates, weights = zip(*self.word_counts.items())
            # Общий знаменатель P сокращается при перенормировке.
            # Деление на максимум сохраняет распределение и устойчивость.
            largest = max(weights)
            weights = [(weight / largest) ** (1 / temperature)
                       for weight in weights]
            generated.append(rng.choices(candidates, weights=weights, k=1)[0])
        return " ".join(generated)

    def logprob(self, text):
        words = tokenize(text) if isinstance(text, str) else list(text)
        terms = []
        for previous, word in zip(words, words[1:]):
            probability = self.prob(previous, word)
            if probability == 0:
                return -math.inf
            terms.append(math.log(probability))
        return math.fsum(terms)

In [ ]:
small = BigramModel().fit(tokenize("я люблю чай я люблю кофе"))
print("P(люблю | я) =", small.prob("я", "люблю"))
print("P(чай | люблю) =", small.prob("люблю", "чай"))

assert small.prob("я", "люблю") == 1
assert small.prob("люблю", "чай") == 0.5
assert small.word_counts["кофе"] == 1
assert small.totals.get("кофе", 0) == 0

P(люблю | я) = 1.0
P(чай | люблю) = 0.5


In [ ]:
model = BigramModel().fit(train)
print("Разных слов в обучении:", len(model.word_counts))
print("Разных пар:", sum(len(row) for row in model.bigrams.values()))

# Для этой таблицы и генерации смотрю обычные частоты без сглаживания.

print("\nЧто чаще всего идёт после слова 'князь':")
for word, count in model.bigrams["князь"].most_common(10):
    print(f"{word:15} {count:5}  P = {model.prob('князь', word):.4f}")

Разных слов в обучении: 45323
Разных пар: 236926

Что чаще всего идёт после слова 'князь':
андрей            775  P = 0.5867
василий           147  P = 0.1113
багратион          36  P = 0.0273
николай            32  P = 0.0242
ипполит            23  P = 0.0174
и                  17  P = 0.0129
не                 13  P = 0.0098
в                   8  P = 0.0061
был                 8  P = 0.0061
болконский          7  P = 0.0053


## Генерация и температура

Сначала генерирую без сглаживания (alpha=0), выбирая продолжение по частотам
из строки предыдущего слова; это то же распределение, что и prob.
Температура заменяет веса на P1/T с последующей перенормировкой.
Если у слова вообще нет исходящих переходов, без сглаживания использую
частоты слов обучения как запасное распределение; при положительном alpha
пустая строка уже задаёт равномерное распределение по словарю.
Фиксированный seed делает примеры воспроизводимыми.

In [ ]:
starts = ["князь", "пьер", "наташа", "война", "человек"]
examples = []
for i, start in enumerate(starts):
    result = model.generate(start, length=30, seed=182648912 + i)
    assert len(result.split()) == 30
    examples.append(result)
    print(f"{i + 1}. {result}\n")

1. князь андрей приехал адъютант просит поцеловать сыну указывая ею восхищался прежде всего шла о себе что-нибудь не могла этого молодого солдата и поехал по полю ему что эта о счастии

2. пьер княжна оглянулась в дверь отворилась почтительно недоумевающих устремленных на нее есть порядок в россии о чем-нибудь свою жизнь и слушал семь лет заплатит этот момент зацепил рычаг и чувствовал

3. наташа сбросила свою душевную силу своего смеха возбуждаемого черноватым гусарским видом как того что говорили вспоминая слова иногда являвшихся в шубах и заметив неестественность его чувству ростова в брюнне и

4. война кончена сказал адъютант и пройдя мимо французской кавалерии действовать в петербург искать другого спасения от своего бурого николай подошел к армии подтверждал другой старик подъехал к церкви где людские

5. человек велел ехать дорогой в которое пьер покорно стал садиться за станком и кончено но в силах понимать слова пока хотя и составлением отдела права наказывать и в разъезды и



Теперь оставлю то же начальное слово и
seed, но поменяю температуру.
При T=0.5 частые переходы должны стать ещё заметнее, а при T=2 — редкие будут выпадать чаще.

In [ ]:
for temperature in (0.5, 1, 2):
    result = model.generate("князь", length=30, temperature=temperature, seed=182648912)
    assert len(result.split()) == 30
    print(f"T = {temperature}\n{result}\n")

T = 0.5
князь андрей был очень понимаю сказал он не двигаясь ничего не будет с ним и с ним и не могла плакать о том что он уже не было ему что

T = 1
князь андрей приехал адъютант просит поцеловать сыну указывая ею восхищался прежде всего шла о себе что-нибудь не могла этого молодого солдата и поехал по полю ему что эта о счастии

T = 2
князь сказал даву для генерального сражения трех началах вещей на ногу и пронеслась туча которую лакей хотел встать но с несвойственной ему осипом алексеевичем в соне осмотрела ее воображения и



При низкой температуре заметнее привычные сочетания и повторы, при высокой текст разнообразнее.
Но модель каждый раз помнит только одно предыдущее слово.

Возьму первое подходящее предложение из отложенной части: от 12 до 30 слов.
Первый фрагмент пропускаю, потому что граница 90/10 могла пройти внутри предложения.
Для сравнения перемешаю те же слова.

In [ ]:
fragments = re.split(r"[.!?]+(?:\s+|$)", test_text)
sentence = next(part.strip() for part in fragments[1:]
                if 12 <= len(tokenize(part)) <= 30)

normal = tokenize(sentence)
mixed = normal.copy()
random.Random(55).shuffle(mixed)
assert Counter(normal) == Counter(mixed)

print("Исходное:", sentence)
print("Перемешанное:", " ".join(mixed))
print("\nlog P исходного:", model.logprob(normal))
print("log P перемешанного:", model.logprob(mixed))

Исходное: на другой день были у фельдмаршала обед и бал, которые государь удостоил своим присутствием
Перемешанное: обед фельдмаршала которые и своим бал на государь удостоил присутствием у день были другой

log P исходного: -inf
log P перемешанного: -inf


## Почему нужно сглаживание

В обоих предложениях есть пары, которых не было в обучении: без сглаживания
их вероятность равна нулю, а $\log 0 = -\infty$.
Поэтому обе суммы равны $-\infty$ и сравнить их по величине нельзя.
Сглаживание Лапласа добавляет $\alpha$ к каждому счётчику:

$$P(w\mid v) = \frac{c(v,w)+\alpha}{c(v)+\alpha V}.$$

Здесь $c(v)$ — число исходящих переходов, $V$ — размер словаря обучения
плюс `<unk>`. Неизвестные слова не расширяют словарь, а отображаются в `<unk>`.
При положительном $\alpha$ все такие вероятности положительны,
а неизвестный контекст даёт равномерное распределение $1/V$.

In [ ]:
model.alpha = 0.1
normal_score = model.logprob(normal)
mixed_score = model.logprob(mixed)
print("log P исходного:", round(normal_score, 4))
print("log P перемешанного:", round(mixed_score, 4))
print("Разница:", round(normal_score - mixed_score, 4))

log P исходного: -111.1398
log P перемешанного: -129.1323
Разница: 17.9925


Теперь оценки конечные, и исходное предложение получило большую вероятность.
В нём чаще встречаются привычные соседние слова. Это не гарантирует правильное сравнение любых
двух фраз: порядок целого предложения и его смысл биграммы не учитывают.

## Перплексия

Перплексия (от англ. perplexity — растерянность, замешательство) — это метрика в машинном обучении и теории информации, которая показывает, насколько хорошо вероятностная языковая модель предсказывает текст.

Как она работает

• Суть метрики: Интуитивно перплексию можно представить как среднее число равновероятных вариантов, между которыми «колеблется» или сомневается модель на каждом шаге при предсказании следующего токена (слова или подслова).
• Правило «чем меньше, тем лучше»: Чем ниже перплексия, тем точнее и увереннее модель понимает текст и тем меньше она «удивляется» новым словам из тестовой выборки.
• Идеальное значение: Перплексия, равная 1, означала бы абсолютную предсказуемость (чего на практике в естественных языках не бывает). Если модель совсем ничего не знает и дает равномерное распределение по всему словарю размера K, то её перплексия будет равна K.

Ограничения метрики

• Зависимость от словаря: Сравнивать перплексию можно только у тех моделей, которые используют одинаковый токенизатор и одинаковый тестовый набор данных.
• Не равна полезности: Низкая перплексия говорит лишь о том, что модель хорошо улавливает статистику языка, но она не гарантирует отсутствие фактических ошибок или «галлюцинаций» в ответах.

$$
PPL = \exp\left(-\frac{\log P}{N}\right).
$$

Здесь $N$ — число переходов, то есть `len(test) - 1`, потому что первое слово не оцениваем.
Пару через границу обучения и проверки тоже не используем. Чем меньше перплексия,
тем выше средняя вероятность следующего слова на этом же отложенном тексте.

In [ ]:
def perplexity(model, words):
    words = tokenize(words) if isinstance(words, str) else list(words)
    if len(words) < 2:
        raise ValueError("Для перплексии нужен хотя бы один переход")
    return math.exp(-model.logprob(words) / (len(words) - 1))

small = BigramModel(alpha=0).fit(tokenize("я люблю чай я люблю кофе"))
assert math.isclose(perplexity(small, tokenize("я люблю чай")), math.sqrt(2))

results = []
print(f"{'alpha':>7} {'перплексия':>14}")
for alpha in (1, 0.1, 0.01):
    model.alpha = alpha
    value = perplexity(model, test)
    results.append((alpha, value))
    print(f"{alpha:7g} {value:14.4f}")

best_alpha, best_ppl = min(results, key=lambda row: row[1])
print(f"\nЛучший результат: alpha={best_alpha}, PPL={best_ppl:.4f}")
model.alpha = 0.1

  alpha     перплексия
      1     15769.4921
    0.1      7756.9508
   0.01      5318.4374

Лучший результат: alpha=0.01, PPL=5318.4374


Среди этих трёх значений лучше получилось alpha=0.01.
При большом alpha добавка alpha × V в знаменателе велика, поэтому распределения становятся
слишком похожи на равномерные и частые переходы теряют преимущество.
Но делать alpha сколь угодно маленьким тоже нельзя: новые пары будут получать слишком малую вероятность.

In [ ]:
unknown_count = sum(word not in model.word_counts for word in test)
print("Неизвестных слов в проверке:", unknown_count)
print(f"Их доля: {unknown_count / len(test):.2%}")

# Проверяю, что оценка не добавила слова из test в обучающие.
assert model.word_counts == Counter(train)
assert set(model.vocab) == set(train) | {"<unk>"}
assert sum(model.totals.values()) == len(train) - 1

# На маленьком тексте проверяю формулу, нормировку и неизвестные слова.
check = BigramModel(alpha=0.1).fit(tokenize("я люблю чай я люблю кофе"))
assert math.isclose(check.prob("люблю", "чай"), 1.1 / 2.5)
for context in ("люблю", "кофе", "неизвестныйконтекст"):
    assert math.isclose(sum(check.prob(context, word) for word in check.vocab), 1.0)
assert check.prob("я", "неизвестноеслово") == check.prob("я", "<unk>")
assert math.isfinite(check.logprob("совсем неизвестные слова"))
assert check.logprob("я люблю чай") == check.logprob(tokenize("я люблю чай"))
assert len(check.generate("неизвестноеслово", length=30, seed=7).split()) == 30
assert len(tokenize(check.generate("неизвестноеслово", length=30, seed=7))) == 30
check.alpha = 0
assert check.logprob("я кофе") == -math.inf
assert len(check.generate("кофе", length=30, seed=7).split()) == 30
assert model.generate("князь", seed=7) == model.generate("князь", seed=7)

Неизвестных слов в проверке: 3496
Их доля: 8.07%
Проверки формул, нормировки, неизвестных слов и генерации пройдены.


## Выводы

Биграммная модель учится простым подсчётом пар, но уже может выдавать узнаваемые сочетания слов.
Смысл быстро теряется, потому что контекст ограничен одним словом, а пунктуацию я удалил.
Температура управляет разнообразием, сглаживание решает проблему нулевых вероятностей.
Неизвестные слова объединены в `<unk>`, поэтому модель не различает их между собой.
Значения alpha сравнивались на одном отложенном отрезке; это результат для данного корпуса,
а не универсальное лучшее значение для любого текста.